# 06 — Bet Sizing

**AFML Chapter 10** — López de Prado (2018)

A profitable model is necessary but not sufficient. How you *size* each
bet determines whether you capture that edge or blow up.

This notebook demonstrates:
1. **Sigmoid sizing** — map raw signal → position size via tunable sigmoid
2. **Probability → bet size** — translate predicted P(profit) into sizes
3. **Signal discretization** — reduce noise via bucketing
4. **Concurrency budgeting** — scale sizes by max concurrent bets

---

In [ ]:
from _setup import *
from afml.labeling import daily_volatility, make_events, triple_barrier_labels
from afml.sample_weights import sample_weight_by_return, normalize_weights
from afml.fracdiff import frac_diff_log
from afml.cross_validation import cv_score
from afml.bet_sizing import (
    bet_size_sigmoid,
    bet_size_from_prob,
    discrete_signal,
    avg_active_signals,
    max_concurrent_signals,
)
from validation.cv import PurgedKFold

from sklearn.ensemble import RandomForestClassifier
from scipy.stats import norm

## 1. Sigmoid bet sizing

The sigmoid function maps any signal to (-1, 1).  Parameter *w*
controls aggressiveness: higher w → more binary (all-in) bets.

In [ ]:
x = pd.Series(np.linspace(-2, 2, 500))

fig, ax = plt.subplots(figsize=(10, 5))
for w, ls in [(1, "--"), (3, "-."), (10, "-"), (50, ":")]:
    ax.plot(x, bet_size_sigmoid(x, w=w), ls=ls, lw=1.5, label=f"w = {w}")

ax.axhline(0, color=GRAY, lw=0.5)
ax.axvline(0, color=GRAY, lw=0.5)
ax.set_xlabel("Raw signal")
ax.set_ylabel("Position size")
ax.set_title("Sigmoid bet sizing: position size vs raw signal", fontweight="bold")
ax.legend(fontsize=10)
fig.tight_layout()
fig.savefig(str(ARTIFACTS_DIR / "06_sigmoid_sizing.png"), dpi=150, bbox_inches="tight")
plt.show()

## 2. Probability-based bet sizing on BTC

Train a Random Forest to predict triple-barrier outcomes,
then convert predicted probabilities into position sizes.

In [ ]:
panel = load_daily_bars()
btc = panel[panel["symbol"] == "BTC-USD"].sort_values("ts").drop_duplicates("ts", keep="last")
close = btc.set_index("ts")["close"]

vol = daily_volatility(close, span=20)
events = make_events(close, vol, holding_periods=10)
tb = triple_barrier_labels(close, events, pt_sl=(2.0, 2.0))
tb["y"] = (tb["label"] == 1).astype(int)

feat = pd.DataFrame(index=tb.index)
feat["ret_1d"] = np.log(close / close.shift(1)).reindex(tb.index)
feat["ret_5d"] = np.log(close / close.shift(5)).reindex(tb.index)
feat["ret_21d"] = np.log(close / close.shift(21)).reindex(tb.index)
feat["vol_20d"] = vol.reindex(tb.index)
feat["fracdiff"] = frac_diff_log(close, d=0.3).reindex(tb.index)

df = feat.join(tb[["y", "t1"]]).dropna()
X = df[feat.columns].values
y = df["y"].values
t1 = pd.Series(df["t1"].values, index=df.index)
sw = normalize_weights(sample_weight_by_return(t1, close)).reindex(df.index).values

# Walk-forward: train on first 70%, predict last 30%
split = int(len(df) * 0.7)
X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]
sw_train = sw[:split]
test_dates = df.index[split:]

rf = RandomForestClassifier(n_estimators=200, max_depth=4, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train, sample_weight=sw_train)

prob = pd.Series(rf.predict_proba(X_test)[:, 1], index=test_dates)
pred_side = pd.Series(np.where(prob > 0.5, 1, -1), index=test_dates)

print(f"OOS period: {test_dates[0].date()} to {test_dates[-1].date()}")
print(f"Mean predicted P(profit): {prob.mean():.3f}")
print(f"OOS accuracy: {(rf.predict(X_test) == y_test).mean():.1%}")

In [ ]:
# Compute bet sizes
size_prob = bet_size_from_prob(prob, pred_side)
size_sig = bet_size_sigmoid(pd.Series(prob.values * 2 - 1, index=prob.index), w=10)

fig, axes = plt.subplots(2, 2, figsize=(14, 8))

# Predicted probabilities
ax = axes[0, 0]
ax.hist(prob, bins=30, color=NAVY, alpha=0.7, edgecolor="white")
ax.axvline(0.5, color=RED, ls="--", lw=1)
ax.set_title("Predicted P(profit)", fontweight="bold")
ax.set_xlabel("Probability")

# Bet sizes from prob
ax = axes[0, 1]
ax.hist(size_prob, bins=30, color=TEAL, alpha=0.7, edgecolor="white")
ax.set_title("Bet size (from probability)", fontweight="bold")
ax.set_xlabel("Position size")

# Bet sizes from sigmoid
ax = axes[1, 0]
ax.hist(size_sig, bins=30, color=GOLD, alpha=0.7, edgecolor="white")
ax.set_title("Bet size (sigmoid, w=10)", fontweight="bold")
ax.set_xlabel("Position size")

# Bet sizes over time
ax = axes[1, 1]
ax.plot(test_dates, size_prob, color=TEAL, alpha=0.6, lw=0.5, label="From prob")
ax.plot(test_dates, size_sig, color=GOLD, alpha=0.6, lw=0.5, label="Sigmoid")
ax.axhline(0, color=GRAY, lw=0.5)
ax.set_title("Bet sizes over time", fontweight="bold")
ax.set_ylabel("Position size")
ax.legend(fontsize=9)

fig.suptitle("BTC: Probability → Bet Size (OOS)", fontweight="bold", fontsize=13)
fig.tight_layout()
fig.savefig(str(ARTIFACTS_DIR / "06_bet_sizes_btc.png"), dpi=150, bbox_inches="tight")
plt.show()

## 3. Signal discretization

Discretizing reduces whipsawing — small changes in probability
don't flip the position.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, step in zip(axes, [0.05, 0.1, 0.2]):
    disc = discrete_signal(size_prob, step_size=step)
    ax.plot(test_dates, size_prob, color=GRAY, alpha=0.3, lw=0.5, label="Continuous")
    ax.step(test_dates, disc, color=NAVY, lw=1, where="mid", label=f"Step={step}")
    ax.axhline(0, color=GRAY, lw=0.5)
    ax.set_title(f"Step size = {step}", fontweight="bold")
    ax.legend(fontsize=8)
    ax.set_ylabel("Position size")

fig.suptitle("Discretized bet sizes reduce turnover", fontweight="bold", fontsize=12)
fig.tight_layout()
plt.show()

## 4. Concurrency budgeting

When multiple bets are active simultaneously, we must budget
position sizes so gross exposure stays bounded.

In [ ]:
max_c = max_concurrent_signals(t1)
print(f"Max concurrent active bets: {max_c}")
print(f"Budget per bet: 1/{max_c} = {1/max_c:.3f}")
print(f"\nWith budgeting, max |bet size| should be {1/max_c:.3f}")

# Apply budget
size_budgeted = size_prob / max_c

fig, ax = plt.subplots(figsize=(12, 4))
ax.fill_between(test_dates, size_prob, alpha=0.2, color=TEAL, label="Unbudgeted")
ax.fill_between(test_dates, size_budgeted, alpha=0.4, color=NAVY, label=f"Budgeted (÷{max_c})")
ax.axhline(0, color=GRAY, lw=0.5)
ax.set_ylabel("Position size")
ax.set_title(f"Concurrency budgeting: max {max_c} concurrent bets", fontweight="bold")
ax.legend(fontsize=9)
fig.tight_layout()
plt.show()

## 5. Summary

**Key takeaways from AFML Chapter 10:**

| Technique | Use Case | Trade-off |
|-----------|----------|----------|
| Sigmoid | Map raw signals to sizes | `w` controls aggressiveness |
| Prob → CDF | Size by prediction confidence | Requires calibrated probabilities |
| Discretization | Reduce turnover/whipsaw | Loses granularity |
| Concurrency budget | Cap gross exposure | Conservative in high-conviction periods |

**For production:** Use probability-based sizing with concurrency budgeting
and discretization (step_size = 0.1–0.2) to balance signal quality with
turnover costs.

---

*Next: [07_structural_breaks.ipynb](07_structural_breaks.ipynb) — Structural Breaks & Entropy (AFML Ch 14–15)*